In [56]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/winter-2026-machine-learning-competition/sample_submission.csv
/kaggle/input/competitions/winter-2026-machine-learning-competition/train.csv
/kaggle/input/competitions/winter-2026-machine-learning-competition/test.csv


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, roc_auc_score

train = pd.read_csv("/kaggle/input/competitions/winter-2026-machine-learning-competition/train.csv")
test = pd.read_csv("/kaggle/input/competitions/winter-2026-machine-learning-competition/test.csv")

# Feature engineering
def feature_engineering(df):
    df = df.copy()

    # Original strong block
    df["X14_sq"] = df["X14"] ** 2
    df["X14_abs"] = np.abs(df["X14"])
    df["X14_X17"] = df["X14"] * df["X17"]
    df["X14_X10"] = df["X14"] * df["X10"]
    df["X14_X4"] = df["X14"] * df["X4"]
    df["X14_sign"] = np.sign(df["X14"])

    df["X1_abs"] = np.abs(df["X1"])
    df["X10_abs"] = np.abs(df["X10"])
    df["X17_sq"] = df["X17"] ** 2
    df["X14_X17_ratio"] = df["X14"] / (np.abs(df["X17"]) + 1e-3)
    df["X14_X10_ratio"] = df["X14"] / (np.abs(df["X10"]) + 1e-3)

    df["X16_sq"] = df["X16"] ** 2
    df["X16_abs"] = np.abs(df["X16"])
    df["X16_X14"] = df["X16"] * df["X14"]
    df["X16_X17"] = df["X16"] * df["X17"]
    df["X16_X10"] = df["X16"] * df["X10"]

    df["mag_14_17"] = np.sqrt(df["X14"]**2 + df["X17"]**2)
    df["mag_14_4"] = np.sqrt(df["X14"]**2 + df["X4"]**2)
    df["mag_14_10"] = np.sqrt(df["X14"]**2 + df["X10"]**2)

    # Earlier helpful block
    df["X19_log"] = np.log1p(np.abs(df["X19"]))
    df["X19_X6"] = df["X19"] * df["X6"]
    df["X19_X11"] = df["X19"] * df["X11"]
    df["X19_X3"] = df["X19"] * df["X3"]

    # New block from hard/easy positive analysis
    df["X22_log"] = np.log1p(np.abs(df["X22"]))
    df["X21_log"] = np.log1p(np.abs(df["X21"]))
    df["X7_log"] = np.log1p(np.abs(df["X7"]))

    df["X22_X19"] = df["X22"] * df["X19"]
    df["X22_X11"] = df["X22"] * df["X11"]
    df["X21_X19"] = df["X21"] * df["X19"]
    df["X7_X19"] = df["X7"] * df["X19"]
    df["X1_X22"] = df["X1"] * df["X22"]
    df["X2_X22"] = df["X2"] * df["X22"]

    
    
    
    return df


# Prepare features/target
y = train["Label"]

X = feature_engineering(train).drop(columns=["Label", "Time"], errors="ignore")
X_test = feature_engineering(test).drop(columns=["id", "Time"], errors="ignore")


# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Resample training data only
smotetomek = SMOTETomek(
    sampling_strategy=0.04,
    random_state=42
)

X_train_res, y_train_res = smotetomek.fit_resample(X_train, y_train)

print("Original training shape:", X_train.shape)
print("After SMOTETomek:", X_train_res.shape)

# =========================
# Train model
# =========================
model = XGBClassifier(
    n_estimators=1800,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=1,
    reg_alpha=0.5,
    reg_lambda=1.5,
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_res, y_train_res)


# Validation predictions
val_preds = model.predict_proba(X_val)[:, 1]


# Validation metrics
val_auc_full = roc_auc_score(y_val, val_preds)
val_auc_partial = roc_auc_score(y_val, val_preds, max_fpr=0.01)

print("Validation ROC AUC:", val_auc_full)
print("Validation ROC AUC (max_fpr=0.01):", val_auc_partial)


# Retrain on full training data
smotetomek_full = SMOTETomek(
    sampling_strategy=0.04,
    random_state=42
)

X_res_full, y_res_full = smotetomek_full.fit_resample(X, y)

print("Original full training shape:", X.shape)
print("After full SMOTETomek:", X_res_full.shape)

final_model = XGBClassifier(
    n_estimators=1800,
    max_depth=6,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=1,
    reg_alpha=0.5,
    reg_lambda=1.5,
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_res_full, y_res_full)


# Predict test set
test_preds = final_model.predict_proba(X_test)[:, 1]


# Submission
submission = pd.DataFrame({
    "id": test["id"],
    "Label": pd.Series(test_preds).rank(pct=True)
})
submission.to_csv("/kaggle/working/submission_smotetomek_features.csv", index=False)
print("Submission file saved as submission_smotetomek_features.csv") 

Original training shape: (136711, 61)
After SMOTETomek: (141921, 61)
Validation ROC AUC: 0.9825312151943255
Validation ROC AUC (max_fpr=0.01): 0.9399252156988314
Original full training shape: (170889, 61)
After full SMOTETomek: (177390, 61)
Submission file saved as submission_smotetomek_features.csv
